In [1]:
import os, re, glob, time, subprocess, pythoncom, psutil
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, date
from pathlib import Path
from IPython.display import HTML, display
import polars as pl

# ══════════════════════════════════════════════════════════════════════════════
# PATHS
# ══════════════════════════════════════════════════════════════════════════════
first_glob     = os.path.expanduser("~").replace("\\", "/")
CAPTURE_FOLDER = f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/CAPTURE/current_agent"
HC_PARQUET     = f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/Resources/hc_extend_combination.parquet"
MASTER_ROSTER  = f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Schedule/Schedule (Ops version)/2026/Master_Schedule_Merged.xlsx"

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════
DISPLAY_NOTEBOOK = True
SEND_EMAIL       = False

EMAIL_TO = "huuchinh.nguyen@concentrix.com"
EMAIL_CC = "huuchinh.nguyen@concentrix.com"

TARGET_PLANNED   = 4.0
TARGET_UNPLANNED = 6.0
TARGET_SHRINKAGE = 10.0
TARGET_ATD       = 90.0

# None = auto detect | "0600-1500" | "1100-2000" | "2200-0700"
REPORT_SHIFT = "2000-0500" #None

# "email": ("CODE", "HALF_or_None", "Reason", "Remark")
LEAVE_OVERRIDES = {
    # "ngocquynhanh.pham@concentrix.com": ("AB", None, "Health issue", None),
    # "mythanh.lam@concentrix.com":       ("AB", None, "Termination",  None),
}

SHIFT_LIST = [
    "0500-1400","0600-1500","0700-1600","0900-1800",
    "1000-1900","1100-2000","1200-2100","1300-2200",
    "2000-0500","2100-0600","2200-0700",
]
LEAVE_SHIFT_CODES = ["AL","CO","LWP","AB","SL","HAL","HLWP","HAB","HSL","CMLF","LATE"]
KEEP_LOBS         = ["Lodging","Non_Lodging"]

# ══════════════════════════════════════════════════════════════════════════════
# SHIFT HELPERS
# ══════════════════════════════════════════════════════════════════════════════
def parse_shift_dt(shift_str, ref_date):
    if not shift_str or not isinstance(shift_str, str): return None
    parts = shift_str.strip().split("-")
    if len(parts) != 2: return None
    try:
        s = datetime.combine(ref_date, datetime.strptime(parts[0].zfill(4), "%H%M").time())
        e = datetime.combine(ref_date, datetime.strptime(parts[1].zfill(4), "%H%M").time())
        if e <= s: e += timedelta(days=1)
        return s, e
    except: return None

def is_night(shift_str):
    try: return int(str(shift_str).split("-")[0][:2]) >= 18
    except: return False

def get_report_scope():
    now    = datetime.now()
    today  = now.date()
    yester = today - timedelta(days=0)

    if REPORT_SHIFT:
        sh = REPORT_SHIFT
    else:
        try:
            _raw = pd.read_excel(MASTER_ROSTER, sheet_name="Sheet1", dtype=str)
            _raw.columns = [str(c) for c in _raw.columns]
            _info = ["IEX ID","OracleID","Email","Employee Name","Status"]
            _dcols = [c for c in _raw.columns if re.match(r"\d{4}-\d{2}-\d{2}", c)]

            _long = (
                _raw[_info + _dcols]
                .melt(id_vars=_info, value_vars=_dcols,
                      var_name="Scheduled_Date", value_name="Roster_Shift")
            )
            _long["Scheduled_Date"] = pd.to_datetime(
                _long["Scheduled_Date"], errors="coerce").dt.date

            _today_shifts = set(
                _long[
                    (_long["Scheduled_Date"] == today) &
                    (_long["Status"] == "Active") &
                    (_long["Roster_Shift"].notna()) &
                    (~_long["Roster_Shift"].isin(
                        ["OFF","Termination","HO","null","nan",""]))
                ]["Roster_Shift"].dropna().unique()
            )

            available_shifts = [s for s in SHIFT_LIST if s in _today_shifts]
            print(f"✓ Available shifts today: {available_shifts}")
        except Exception as e:
            print(f"⚠️ Could not pre-load roster for shift detection: {e}")
            available_shifts = SHIFT_LIST

        cands = []
        for s in available_shifts:
            sh_h = int(s.split("-")[0][:2])
            ref  = yester if sh_h >= 18 else today
            p    = parse_shift_dt(s, ref)
            if p and p[0] <= now:
                cands.append((p[0], s))

        if not cands:
            day_shifts = [s for s in available_shifts
                          if int(s.split("-")[0][:2]) < 18]
            sh = day_shifts[0] if day_shifts else SHIFT_LIST[0]
            print(f"⚠️ No shift started yet, using earliest: {sh}")
        else:
            sh = sorted(cands)[-1][1]

    sh_h        = int(sh.split("-")[0][:2])
    night       = sh_h >= 18
    roster_date = yester if night else today

    if night:
        scope = SHIFT_LIST[:]
    else:
        rep_h = int(sh.split("-")[0][:2])
        scope = [s for s in SHIFT_LIST
                 if int(s.split("-")[0][:2]) <= rep_h
                 and int(s.split("-")[0][:2]) < 18]

    scope = scope + LEAVE_SHIFT_CODES
    return sh, roster_date, scope

report_shift, roster_date, shifts_in_scope = get_report_scope()
report_now    = datetime.now()
report_date_s = report_now.strftime("%d-%b-%Y")
EMAIL_SUBJECT = (f"Expedia VN Attendance Report as of the shift "
                 f"{report_shift} on {report_date_s} (VNT)")

print(f"✓ Report shift   : {report_shift}")
print(f"✓ Roster date    : {roster_date}")
print(f"✓ Shifts in scope: {shifts_in_scope}")
print(f"✓ Subject        : {EMAIL_SUBJECT}")

# ══════════════════════════════════════════════════════════════════════════════
# LOAD MASTER ROSTER
# ══════════════════════════════════════════════════════════════════════════════
print("📂 Loading Master Schedule...")
raw = pd.read_excel(MASTER_ROSTER, sheet_name="Sheet1", dtype=str)
raw.columns = [str(c) for c in raw.columns]

info_cols = ["IEX ID","OracleID","Email","Employee Name","Status"]
date_cols = [c for c in raw.columns if re.match(r"\d{4}-\d{2}-\d{2}", c)]

roster_long = (
    raw[info_cols + date_cols]
    .melt(id_vars=info_cols, value_vars=date_cols,
          var_name="Scheduled_Date", value_name="Roster_Shift")
    .rename(columns={"IEX ID":"IEX","OracleID":"Emp ID","Employee Name":"Agent Name"})
)
roster_long["Scheduled_Date"] = pd.to_datetime(
    roster_long["Scheduled_Date"], errors="coerce").dt.date
roster_long["IEX"]    = pd.to_numeric(roster_long["IEX"],    errors="coerce")
roster_long["Emp ID"] = pd.to_numeric(roster_long["Emp ID"], errors="coerce")
roster_long["Roster_Shift"] = roster_long["Roster_Shift"].replace("-", np.nan)

def compute_roster_dt(df):
    rows_s, rows_e, nights = [], [], []
    for _, r in df.iterrows():
        ref = r["Scheduled_Date"]
        sh  = r["Roster_Shift"]
        p   = parse_shift_dt(sh, ref) if pd.notna(sh) else None
        rows_s.append(p[0] if p else pd.NaT)
        rows_e.append(p[1] if p else pd.NaT)
        nights.append(1 if pd.notna(sh) and is_night(sh) else 0)
    df = df.copy()
    df["Datetime_Start_Shift"] = rows_s
    df["Datetime_End_Shift"]   = rows_e
    df["Night_Shift"]          = nights
    return df

roster_long = compute_roster_dt(roster_long)

today_roster = roster_long[
    (roster_long["Scheduled_Date"] == roster_date) &
    (~roster_long["Status"].isin(["Terminated",""])) &
    (roster_long["Roster_Shift"].notna()) &
    (~roster_long["Roster_Shift"].isin(["OFF","Termination","HO","null","nan",""])) &
    (
        roster_long["Roster_Shift"].isin(shifts_in_scope) |
        roster_long["Roster_Shift"].isin(LEAVE_SHIFT_CODES)
    )
].copy().reset_index(drop=True)

print(f"✓ Roster agents in scope: {len(today_roster)}")

# ══════════════════════════════════════════════════════════════════════════════
# LOAD HC EXTEND — latest record per agent ≤ roster_date
# ══════════════════════════════════════════════════════════════════════════════
print("📂 Loading HC Extend parquet...")
hc_pl = (
    pl.read_parquet(HC_PARQUET)
    .with_columns(pl.col("Date").cast(pl.Date, strict=False))
    .filter(pl.col("Date") <= pl.lit(roster_date))
    .sort("Date", descending=True)
    .unique(subset=["Email Id"], keep="first")
)
hc = hc_pl.to_pandas()
print(f"✓ HC Extend rows (latest per agent ≤ {roster_date}): {len(hc)}")

email_col_hc = "Email Id"
if email_col_hc not in hc.columns:
    email_col_hc = next(
        (c for c in hc.columns
         if "email" in c.lower() and "supervisor" not in c.lower()), None)
    print(f"⚠️ Fallback email col: {email_col_hc}")

HC_INFO_COLS = ["Supervisor Name","LOB","Wave","Alias","OracleID"]
if email_col_hc and not hc.empty:
    hc_map = (
        hc[[email_col_hc] + [c for c in HC_INFO_COLS if c in hc.columns]]
        .drop_duplicates(subset=[email_col_hc], keep="last")
        .rename(columns={email_col_hc: "_hc_email"})
    )
    hc_map["_hc_email"] = hc_map["_hc_email"].str.lower().str.strip()
else:
    hc_map = pd.DataFrame(columns=["_hc_email"] + HC_INFO_COLS)

print(f"✓ HC map entries: {len(hc_map)}")

today_roster["_email_key"] = today_roster["Email"].str.lower().str.strip()
today_roster = today_roster.merge(
    hc_map.rename(columns={"_hc_email": "_email_key"}),
    on="_email_key", how="left"
)

if "OracleID_x" in today_roster.columns:
    today_roster["OracleID"] = (today_roster["OracleID_x"]
                                .combine_first(today_roster["OracleID_y"]))
    today_roster.drop(columns=["OracleID_x","OracleID_y"], inplace=True, errors="ignore")
elif "OracleID" not in today_roster.columns:
    today_roster["OracleID"] = today_roster["Emp ID"]

# ══════════════════════════════════════════════════════════════════════════════
# LOB MAPPING
# ══════════════════════════════════════════════════════════════════════════════
def map_lob(lob):
    if pd.isna(lob): return None
    s = str(lob).strip()
    if s == "Support_LG_Nesting": return "Lodging"
    if s == "Support_NL_Nesting": return "Non_Lodging"
    if s == "Lodging":            return "Lodging"
    if s == "Non_Lodging":        return "Non_Lodging"
    return None

if "LOB" in today_roster.columns:
    today_roster["LOB"] = today_roster["LOB"].apply(map_lob)

print(f"✓ LOB: {today_roster['LOB'].value_counts(dropna=False).to_dict()}")

# ══════════════════════════════════════════════════════════════════════════════
# LEAVE OVERRIDES
# ══════════════════════════════════════════════════════════════════════════════
def half_shift_str(roster_shift, half):
    p = parse_shift_dt(roster_shift, roster_date)
    if not p: return roster_shift
    s, e   = p
    half_h = (e - s).total_seconds() / 3600 / 2
    if half == "FHAL": ns, ne = s + timedelta(hours=half_h), e
    else:              ns, ne = s, s + timedelta(hours=half_h)
    return f"{ns.strftime('%H%M')}-{ne.strftime('%H%M')}"

today_roster["Leave"]       = None
today_roster["Final_Shift"] = today_roster["Roster_Shift"]
today_roster["Reason"]      = None
today_roster["Remark"]      = None
today_roster["Start_Shift"] = today_roster["Datetime_Start_Shift"]
today_roster["End_Shift"]   = today_roster["Datetime_End_Shift"]

for raw_email, ov in LEAVE_OVERRIDES.items():
    leave_code = ov[0]
    half       = ov[1] if len(ov) > 1 else None
    reason     = ov[2] if len(ov) > 2 else None
    remark     = ov[3] if len(ov) > 3 else None

    mask = today_roster["_email_key"] == raw_email.lower().strip()
    if not mask.any():
        print(f"  ⚠️ Leave override not found: {raw_email}")
        continue

    today_roster.loc[mask, "Leave"]  = leave_code
    today_roster.loc[mask, "Reason"] = reason
    today_roster.loc[mask, "Remark"] = remark

    if leave_code in ("HAL","HLWP") and half in ("FHAL","SHAL"):
        today_roster.loc[mask, "Final_Shift"] = (
            today_roster.loc[mask, "Roster_Shift"]
            .apply(lambda s: half_shift_str(s, half))
        )
        for idx in today_roster[mask].index:
            p = parse_shift_dt(today_roster.at[idx, "Final_Shift"], roster_date)
            if p:
                today_roster.at[idx, "Start_Shift"] = p[0]
                today_roster.at[idx, "End_Shift"]   = p[1]
    elif leave_code in ("AL","CO","LWP","AB","SL","CMLF","NCNS"):
        today_roster.loc[mask, "Final_Shift"] = leave_code
        today_roster.loc[mask, "Start_Shift"] = pd.NaT
        today_roster.loc[mask, "End_Shift"]   = pd.NaT

print(f"✓ Leave overrides applied: {today_roster['Leave'].notna().sum()} agent(s)")

# ══════════════════════════════════════════════════════════════════════════════
# LOGIN DATA
# ══════════════════════════════════════════════════════════════════════════════
print("📂 Loading login CSVs...")

def load_min_logins():
    today     = roster_date
    yester    = today - timedelta(days=1)
    cut_day   = datetime.combine(today,  datetime.min.time())
    cut_night = datetime.combine(yester, datetime.strptime("1800", "%H%M").time())
    rows      = []

    for f in glob.glob(f"{CAPTURE_FOLDER}/*.csv"):
        try:
            stem  = Path(f).stem
            m     = re.search(r"(\w{3})\s+(\d{1,2})\s+(\d{4})", stem)
            if not m: continue
            fdate = pd.to_datetime(
                f"{m.group(1)} {m.group(2)} {m.group(3)}", format="%b %d %Y").date()
            if fdate not in (today, yester): continue

            tmp      = pd.read_csv(f, dtype=str, encoding="utf-8-sig")
            site_col = next((c for c in tmp.columns
                             if "business" in c.lower() or "location" in c.lower()), None)
            if site_col:
                tmp = tmp[tmp[site_col].str.contains("Ho Chi Minh", na=False, case=False)]

            ecol = next((c for c in tmp.columns if "email" in c.lower()), None)
            tcol = next((c for c in tmp.columns
                         if "login" in c.lower() and "time" in c.lower()), None)
            if not ecol or not tcol: continue

            tmp         = tmp[[ecol, tcol]].copy()
            tmp.columns = ["Agent Email","Login Time"]
            tmp["Login Time"]  = pd.to_datetime(tmp["Login Time"], errors="coerce")
            tmp["Agent Email"] = tmp["Agent Email"].str.lower().str.strip()
            rows.append(tmp.dropna(subset=["Login Time"]))
        except Exception as ex:
            print(f"  ⚠️ Skip {Path(f).name}: {ex}")

    if not rows:
        empty = pd.DataFrame(columns=["Agent Email","Login Time"])
        return empty, empty

    raw       = pd.concat(rows, ignore_index=True)
    day_min   = (raw[raw["Login Time"] >= cut_day]
                 .groupby("Agent Email")["Login Time"].min().reset_index()
                 .rename(columns={"Login Time": "MinLoginTime"}))
    night_min = (raw[raw["Login Time"] >= cut_night]
                 .groupby("Agent Email")["Login Time"].min().reset_index()
                 .rename(columns={"Login Time": "MinLoginTime"}))
    return day_min, night_min

day_min_df, night_min_df = load_min_logins()
print(f"✓ Day logins: {len(day_min_df)} | Night logins: {len(night_min_df)}")

def get_min_login(row):
    email = str(row.get("_email_key", ""))
    src   = night_min_df if is_night(str(row.get("Roster_Shift", ""))) else day_min_df
    m     = src[src["Agent Email"] == email]
    return m.iloc[0]["MinLoginTime"] if not m.empty else pd.NaT

today_roster["MinLoginTime"] = today_roster.apply(get_min_login, axis=1)
print(f"✓ Agents with login: {today_roster['MinLoginTime'].notna().sum()}")

# ══════════════════════════════════════════════════════════════════════════════
# ATD CALCULATIONS — mirror Power Query logic
# ══════════════════════════════════════════════════════════════════════════════
LEAVE_DIRECT     = {"HAL","AL","LWP","AB","SL","CO","CMLF","LATE","NCNS"}
UNPLANNED_LEAVES = {"AB","SL","NCNS","CMLF"}
PLANNED_LEAVES   = {"AL","CO","LWP","HAL","HLWP","HAB","HSL","LATE"}

def _hrs(start, end):
    try:
        d = (pd.Timestamp(end) - pd.Timestamp(start)).total_seconds() / 3600
        return d if d > 0 else None
    except: return None

def calc_hc_schedule(row):
    sh = str(row.get("Roster_Shift", ""))
    if not sh or sh in ("nan","None",""): return None
    if any(x in sh for x in ["HAL","HAB","HLWP","HSL","In Training"]): return 1.0
    if sh in ("AL","CO","LWP","AB","SL","CMLF","LATE"): return 1.0
    h = _hrs(row.get("Datetime_Start_Shift"), row.get("Datetime_End_Shift"))
    if h is not None: return 1.0 if h > 6 else 0.5
    return None

def calc_attendance(row):
    shift = str(row.get("Final_Shift", ""))
    leave = row.get("Leave")
    login = row.get("MinLoginTime")
    if shift in LEAVE_DIRECT: return shift
    if leave and leave in LEAVE_DIRECT: return leave
    if pd.notna(login): return "PR"
    return "NCNS"

def calc_present(row):
    leave = row.get("Leave")
    shift = str(row.get("Final_Shift", ""))
    atd   = row.get("Attendance", "")
    tod   = _hrs(row.get("Start_Shift"), row.get("End_Shift"))
    if tod is None: return None
    if leave is not None and tod < 6:  return 0.5
    if leave is not None and tod >= 6: return 0.0
    if "-" in shift and tod < 6:       return 0.5
    if "-" in shift and tod >= 6:      return 1.0
    if "-" in shift and atd in ("HAL","HAB","HSL","HLWP"): return 0.5
    return None

def calc_planned(row):
    rs    = str(row.get("Roster_Shift", ""))
    leave = row.get("Leave")
    tod   = _hrs(row.get("Datetime_Start_Shift"), row.get("Datetime_End_Shift"))
    if any(x in rs for x in ["HAL","HLWP","HAB","HSL"]): return 0.5
    if rs in ("AL","CO","AB","SL","LWP","CMLF"):          return 1.0
    if leave in ("AL","CO","LWP","HAL","HLWP"):            return 1.0
    if leave in ("AB","SL","CMLF","NCNS"):                 return 0.0
    if tod is None:
        p = parse_shift_dt(rs, roster_date)
        tod = (p[1]-p[0]).total_seconds()/3600 if p else None
    if tod is None: return None
    return 0.5 if tod < 5 else 0.0

def calc_unplanned(row):
    atd   = row.get("Attendance", "")
    leave = row.get("Leave")
    if atd in PLANNED_LEAVES or leave in PLANNED_LEAVES:     return 0.0
    if atd in UNPLANNED_LEAVES or leave in UNPLANNED_LEAVES: return 1.0
    tod = _hrs(row.get("Start_Shift"), row.get("End_Shift"))
    if tod is None: return None
    if tod < 5  and atd != "PR": return 0.5
    if tod >= 5 and atd != "PR": return 1.0
    return 0.0

def calc_late(row):
    login = row.get("MinLoginTime")
    start = row.get("Start_Shift")
    if pd.isna(login) or pd.isna(start): return None
    diff_s = (pd.Timestamp(login) - pd.Timestamp(start)).total_seconds()
    return diff_s if diff_s > 180 else None

def fmt_hms(secs):
    if secs is None or (isinstance(secs, float) and np.isnan(secs)): return "00:00:00"
    s = int(abs(secs)); h, rem = divmod(s, 3600); m, ss = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{ss:02d}"

today_roster["HC Schedule"]  = today_roster.apply(calc_hc_schedule, axis=1)
today_roster["Attendance"]   = today_roster.apply(calc_attendance,  axis=1)
today_roster["Present"]      = today_roster.apply(calc_present,     axis=1)
today_roster["HC Planned"]   = today_roster.apply(calc_planned,     axis=1)
today_roster["HC Unplanned"] = today_roster.apply(calc_unplanned,   axis=1)
today_roster["Late_Secs"]    = today_roster.apply(calc_late,        axis=1)
today_roster["Late_Flag"]    = today_roster["Late_Secs"].notna().astype(int)
today_roster["Login Late"]   = today_roster["Late_Secs"].apply(fmt_hms)

atd_df = today_roster.copy()

all_day_roster = roster_long[
    (roster_long["Scheduled_Date"] == roster_date) &
    (~roster_long["Status"].isin(["Terminated",""])) &
    (roster_long["Roster_Shift"].notna()) &
    (~roster_long["Roster_Shift"].isin(["OFF","Termination","HO","null","nan",""]))
].copy()

# Tính HC Schedule cho tất cả agents trong ngày
all_day_roster["HC Schedule"] = all_day_roster.apply(calc_hc_schedule, axis=1)

# Join LOB để filter Lodging/Non_Lodging
all_day_roster["_email_key"] = all_day_roster["Email"].str.lower().str.strip()
all_day_roster = all_day_roster.merge(
    hc_map.rename(columns={"_hc_email": "_email_key"}),
    on="_email_key", how="left"
)
if "OracleID_x" in all_day_roster.columns:
    all_day_roster.drop(columns=["OracleID_x","OracleID_y"], inplace=True, errors="ignore")

if "LOB" in all_day_roster.columns:
    all_day_roster["LOB"] = all_day_roster["LOB"].apply(map_lob)

# HC_Schedule_Sum per LOB cho cả ngày
hc_schedule_day = (
    all_day_roster[all_day_roster["LOB"].isin(KEEP_LOBS)]
    .groupby("LOB")["HC Schedule"]
    .sum()
    .reset_index()
    .rename(columns={"HC Schedule": "HC_Schedule_Day"})
)
hc_schedule_day_total = all_day_roster[
    all_day_roster["LOB"].isin(KEEP_LOBS)
]["HC Schedule"].sum()

print(f"✓ HC Schedule Day (all shifts): {hc_schedule_day.to_dict('records')}")
print(f"✓ HC Schedule Day Grand Total : {hc_schedule_day_total}")

print(f"✓ ATD computed — {len(atd_df)} agents")
print(f"   PR:{(atd_df['Attendance']=='PR').sum()} "
      f"NCNS:{(atd_df['Attendance']=='NCNS').sum()} "
      f"Leave:{atd_df['Attendance'].isin(list(LEAVE_DIRECT)).sum()}")

# ══════════════════════════════════════════════════════════════════════════════
# SUMMARY
# ══════════════════════════════════════════════════════════════════════════════
def pct(n, d):
    try: return round(float(n)/float(d)*100, 2) if d and d > 0 else None
    except: return None

def build_summary(df, group_cols, schedule_override=None):
    """
    schedule_override: dict {group_val: hc_schedule_day} hoặc float (cho site wise)
    Nếu có override → dùng làm denominator cho Planned/Unplanned/Shrinkage
    """
    agg = {
        "Schedule"    : ("HC Schedule",  "sum"),
        "Present"     : ("Present",       "sum"),
        "HC_Planned"  : ("HC Planned",    "sum"),
        "HC_Unplanned": ("HC Unplanned",  "sum"),
        "Late"        : ("Late_Flag",     "sum"),
    }
    if group_cols:
        g = df.groupby(group_cols, dropna=False).agg(**agg).reset_index()
    else:
        g = pd.DataFrame([{k: df[v[0]].sum() for k, v in agg.items()}])

    def get_denom(row, col):
        if schedule_override is None:
            return row["Schedule"]
        if isinstance(schedule_override, (int, float)):
            return schedule_override
        # dict: key = LOB value
        key = row.get("LOB") if "LOB" in row else None
        return schedule_override.get(key, row["Schedule"]) if key else row["Schedule"]

    g["Planned (%)"]    = g.apply(lambda r: pct(r["HC_Planned"],
                                    get_denom(r, "Schedule")), axis=1)
    g["Unplanned (%)"]  = g.apply(lambda r: pct(r["HC_Unplanned"],
                                    get_denom(r, "Schedule")), axis=1)
    g["Shrinkage (%)"]  = g.apply(lambda r: pct(r["HC_Planned"]+r["HC_Unplanned"],
                                    get_denom(r, "Schedule")), axis=1)
    g["Attendance (%)"] = g.apply(lambda r: pct(r["Present"], r["Schedule"]), axis=1)
    return g.rename(columns={"HC_Planned":"HC Planned","HC_Unplanned":"HC Unplanned"})

def grand_total(df, label_col, total_schedule_day=None):
    nc  = ["Schedule","Present","HC Planned","HC Unplanned","Late"]
    row = {label_col: "Grand Total"}
    for c in nc:
        if c in df.columns: row[c] = df[c].sum()
    hs_denom = total_schedule_day if total_schedule_day else row.get("Schedule", 0)
    hs       = row.get("Schedule", 0)
    pl       = row.get("HC Planned", 0)
    ul       = row.get("HC Unplanned", 0)
    if hs_denom > 0:
        row["Planned (%)"]    = round(pl/hs_denom*100, 2)
        row["Unplanned (%)"]  = round(ul/hs_denom*100, 2)
        row["Shrinkage (%)"]  = round((pl+ul)/hs_denom*100, 2)
    if hs > 0:
        row["Attendance (%)"] = round(row.get("Present", 0)/hs*100, 2)
    return pd.concat([df, pd.DataFrame([row])], ignore_index=True)

atd_lob = (atd_df[atd_df["LOB"].isin(KEEP_LOBS)].copy()
           if "LOB" in atd_df.columns else atd_df.copy())

lob_c = "LOB" if "LOB" in atd_lob.columns else None
sup_c = "Supervisor Name" if "Supervisor Name" in atd_lob.columns else None

sched_override_lob = dict(zip(hc_schedule_day["LOB"],hc_schedule_day["HC_Schedule_Day"]))

# 1. Site wise — denominator = HC_Schedule_Day per LOB
site_sum = build_summary(atd_lob, [lob_c] if lob_c else [],schedule_override=sched_override_lob)
site_sum = grand_total(site_sum, lob_c or "LOB",total_schedule_day=hc_schedule_day_total)

# 2. TL wise — denominator = HC_Schedule_Day per LOB
tl_gc  = [c for c in [lob_c, sup_c] if c]
tl_sum = build_summary(atd_lob, tl_gc,
                       schedule_override=sched_override_lob) if tl_gc else pd.DataFrame()
if not tl_sum.empty:
    tl_sum = grand_total(tl_sum, tl_gc[-1],
                         total_schedule_day=hc_schedule_day_total)

# 3. Shift wise — denominator = HC_Schedule_Day per LOB
sh_gc     = [c for c in [lob_c, "Roster_Shift"] if c and c in atd_lob.columns]
shift_sum = build_summary(atd_lob, sh_gc,
                          schedule_override=None) if sh_gc else pd.DataFrame()
if not shift_sum.empty:
    shift_sum = grand_total(shift_sum, sh_gc[-1],
                            total_schedule_day=hc_schedule_day_total)

# Absenteeism: sort shift "-" theo giờ trước, leave codes sau
def shift_sort_key(shift):
    if pd.isna(shift): return (1, 9999, "")
    s = str(shift)
    if "-" in s:
        try:    return (0, int(s.split("-")[0].zfill(4)), s)
        except: return (0, 9999, s)
    return (1, 0, s)

absent_df = atd_df[
    atd_df["Attendance"].isin(["NCNS","AB","SL","AL","LWP","HAL","CO","CMLF"]) &
    atd_df["LOB"].isin(KEEP_LOBS)
].copy()
absent_df["_sort_shift"] = absent_df["Roster_Shift"].apply(shift_sort_key)
sort_cols = [c for c in [lob_c, sup_c, "_sort_shift", "Agent Name"]
             if c and c in absent_df.columns]
if sort_cols: absent_df = absent_df.sort_values(sort_cols)
absent_df = absent_df.drop(columns=["_sort_shift"], errors="ignore")

print(f"✓ Site:{len(site_sum)} | TL:{len(tl_sum)} | "
      f"Shift:{len(shift_sum)} | Absent:{len(absent_df)}")

# ══════════════════════════════════════════════════════════════════════════════
# STYLE CONSTANTS
# ══════════════════════════════════════════════════════════════════════════════
MET_BG="#d4f4e2"; MET_FG="#1a5c2a"
MISS_BG="#fde8ea"; MISS_FG="#9b1c2a"
WARN_BG="#fff3cd"; WARN_FG="#7a5200"
HDR_DARK="#1a3a5c"; HDR_MID="#1f5c99"
TOT_BG="#1a3a5c"
WHT_ROW="#ffffff"
BANNER_C="#8b0020"
SEC_BADGE_BG = "#e6a817"
SEC_BADGE_FG = "#1a1a1a"
FONT="font-family:Arial,sans-serif;font-size:11px;"
TH_S=(f"{FONT}padding:5px 8px;color:#fff;font-weight:bold;"
      f"white-space:nowrap;text-align:center;"
      f"border:1px solid rgba(255,255,255,0.2);")
TD_S=f"{FONT}padding:4px 8px;border:1px solid #e8e8e8;white-space:nowrap;"
TD_TOT=(f"{FONT}padding:4px 8px;border:1px solid rgba(255,255,255,0.15);"
        f"background:{TOT_BG};color:#fff;font-weight:bold;")

CSS=f"""
body{{margin:0;padding:16px;background:#fff;font-family:Arial,sans-serif}}
.t{{border-collapse:collapse;font-size:11px;white-space:nowrap;width:auto}}
.t thead th{{padding:5px 8px;color:#fff;font-weight:bold;
   text-align:center;border:1px solid rgba(255,255,255,0.2)}}
.t tbody td{{padding:4px 8px;border:1px solid #e8e8e8;
   text-align:left;background:#fff}}
.t tbody tr.tot td{{background:{TOT_BG}!important;color:#fff!important;
   font-weight:bold!important}}
.met{{background:{MET_BG}!important;color:{MET_FG}!important;font-weight:bold!important}}
.miss{{background:{MISS_BG}!important;color:{MISS_FG}!important;font-weight:bold!important}}
.sec-badge{{display:inline-block;font-size:12px;font-weight:bold;
   background:{SEC_BADGE_BG};color:{SEC_BADGE_FG};
   padding:4px 12px;margin:24px 0 4px;border-radius:3px}}
.note{{font-size:10.5px;color:#555;background:#f8f8f8;
   border-left:3px solid {SEC_BADGE_BG};padding:4px 10px;
   margin:0 0 10px;border-radius:0 3px 3px 0}}
"""

# ══════════════════════════════════════════════════════════════════════════════
# HTML HELPERS
# ══════════════════════════════════════════════════════════════════════════════
def fv(v, p=False):
    if v is None or (isinstance(v, float) and np.isnan(v)): return "&#8212;"
    if p: return f"{float(v):.2f}%"
    if isinstance(v, float): return f"{v:,.1f}"
    return str(v)

def _cc(v, tgt, hb=True, em=False):
    if v is None or (isinstance(v, float) and np.isnan(v)): return ("","")
    met = float(v) <= tgt if hb else float(v) >= tgt
    if em:
        s = (f"background:{MET_BG};color:{MET_FG};font-weight:bold;" if met
             else f"background:{MISS_BG};color:{MISS_FG};font-weight:bold;")
        return ("", s)
    return ("met" if met else "miss", "")

def _th(l, bg=HDR_MID):   return f'<th style="{TH_S}background:{bg};">{l}</th>'
def _thl(l, bg=HDR_DARK): return f'<th style="{TH_S}background:{bg};text-align:left;">{l}</th>'

def _tdl(v, tot=False, em=False, bg=WHT_ROW):
    val = str(v) if v is not None and not (isinstance(v,float) and np.isnan(v)) else "&#8212;"
    if tot: return f'<td style="{TD_TOT}text-align:left;">{val}</td>'
    return f'<td style="{TD_S}background:#fff;">{val}</td>'

def _tdn(v, tot=False, em=False, bg=WHT_ROW):
    val = fv(v)
    if tot: return f'<td style="{TD_TOT}text-align:right;">{val}</td>'
    return f'<td style="{TD_S}text-align:right;background:#fff;">{val}</td>'

def _tdp(v, tgt, hb=True, tot=False, em=False, bg=WHT_ROW):
    val = fv(v, True)
    if tot: return f'<td style="{TD_TOT}text-align:right;">{val}</td>'
    cls, inl = _cc(v, tgt, hb, em)
    if em:  return f'<td style="{TD_S}text-align:right;background:#fff;{inl}">{val}</td>'
    return  f'<td class="{cls}" style="{TD_S}text-align:right;background:#fff;">{val}</td>'

def _sec(num, title, note, em=False):
    bs = (f"display:inline-block;font-size:12px;font-weight:bold;"
          f"background:{SEC_BADGE_BG};color:{SEC_BADGE_FG};"   # ← vàng
          f"padding:4px 12px;margin:24px 0 4px;border-radius:3px;")
    ns = (f"{FONT}font-size:10.5px;color:#555;background:#f8f8f8;"
          f"border-left:3px solid {SEC_BADGE_BG};"             # ← viền vàng
          f"padding:4px 10px;margin:0 0 10px;"
          f"border-radius:0 3px 3px 0;display:block;")
    if em:
        return (f'<p style="margin:24px 0 4px;">'
                f'<span style="{bs}">{num}. {title}</span></p>'
                f'<p style="{ns}">{note}</p>')
    return (f'<div style="margin:24px 0 4px;">'
            f'<span class="sec-badge" style="background:{SEC_BADGE_BG};'
            f'color:{SEC_BADGE_FG};">{num}. {title}</span></div>'
            f'<div class="note" style="border-left-color:{SEC_BADGE_BG};">'
            f'{note}</div>')
# col spec: (header, col_key, is_left, target, higher_bad)
SITE_SPEC = [
    ("LOB",           "LOB",            True,  None,             True),
    ("Schedule",      "Schedule",       False, None,             True),
    ("Present",       "Present",        False, None,             True),
    ("HC Planned",    "HC Planned",     False, None,             True),
    ("HC Unplanned",  "HC Unplanned",   False, None,             True),
    ("Late",          "Late",           False, None,             True),
    ("Planned (%)",   "Planned (%)",    False, TARGET_PLANNED,   True),
    ("Unplanned (%)","Unplanned (%)",   False, TARGET_UNPLANNED, True),
    ("Shrinkage (%)","Shrinkage (%)",   False, TARGET_SHRINKAGE, True),
    ("Attendance (%)","Attendance (%)", False, TARGET_ATD,       False),
]
TL_SPEC = [
    ("LOB",           "LOB",            True,  None,             True),
    ("Supervisor",    "Supervisor Name",True,  None,             True),
    ("Schedule",      "Schedule",       False, None,             True),
    ("Present",       "Present",        False, None,             True),
    ("HC Planned",    "HC Planned",     False, None,             True),
    ("HC Unplanned",  "HC Unplanned",   False, None,             True),
    ("Late",          "Late",           False, None,             True),
    ("Login Late",    "Login Late",     True,  None,             True),
    ("Planned (%)",   "Planned (%)",    False, TARGET_PLANNED,   True),
    ("Unplanned (%)","Unplanned (%)",   False, TARGET_UNPLANNED, True),
    ("Shrinkage (%)","Shrinkage (%)",   False, TARGET_SHRINKAGE, True),
    ("Attendance (%)","Attendance (%)", False, TARGET_ATD,       False),
]
SH_SPEC = [
    ("LOB",           "LOB",            True,  None,             True),
    ("Roster Shift",  "Roster_Shift",   True,  None,             True),
    ("Schedule",      "Schedule",       False, None,             True),
    ("Present",       "Present",        False, None,             True),
    ("HC Planned",    "HC Planned",     False, None,             True),
    ("HC Unplanned",  "HC Unplanned",   False, None,             True),
    ("Late",          "Late",           False, None,             True),
    ("Login Late",    "Login Late",     True,  None,             True),
    ("Planned (%)",   "Planned (%)",    False, TARGET_PLANNED,   True),
    ("Unplanned (%)","Unplanned (%)",   False, TARGET_UNPLANNED, True),
    ("Shrinkage (%)","Shrinkage (%)",   False, TARGET_SHRINKAGE, True),
    ("Attendance (%)","Attendance (%)", False, TARGET_ATD,       False),
]

# ══════════════════════════════════════════════════════════════════════════════
# TABLE RENDERERS
# ══════════════════════════════════════════════════════════════════════════════
def render_table(df, spec, em=False):
    tc  = "" if em else 'class="t" '
    col = [(h, k, il, tgt, hb) for h, k, il, tgt, hb in spec if k in df.columns]
    h   = [f'<table {tc}style="border-collapse:collapse;width:auto;{FONT}"><thead><tr>']
    for hdr, _, il, _t, _h in col:
        h.append(_thl(hdr) if il else _th(hdr))
    h.append('</tr></thead><tbody>')

    for i, row in df.iterrows():
        tot = any(str(row.get(c, "")) == "Grand Total"
                  for c in ["LOB","Supervisor Name","Roster_Shift"]
                  if c in df.columns)
        h.append('<tr>' if em else f'<tr class="{"tot" if tot else ""}">')
        for _, k, il, tgt, hb in col:
            v = row.get(k)
            if il:    h.append(_tdl(v, tot, em))
            elif tgt: h.append(_tdp(v, tgt, hb, tot, em))
            else:     h.append(_tdn(v, tot, em))
        h.append('</tr>')
    h.append('</tbody></table>')
    return "".join(h)

def render_absent(df, em=False):
    ACOLORS = {
        "NCNS": (MISS_BG,  MISS_FG),
        "AB":   (MISS_BG,  MISS_FG),
        "HAL":  (WARN_BG,  WARN_FG),
        "AL":   ("#dce8f5","#1a3a5c"),
        "SL":   ("#e8f4fd","#1a3a5c"),
        "CO":   ("#e8f5e9","#1a5c2a"),
        "LWP":  ("#f3e5f5","#6a1b9a"),
        "CMLF": (MISS_BG,  MISS_FG),
    }
    show = ["OracleID","Agent Name","Supervisor Name","LOB",
            "Alias","Wave","Roster_Shift","Attendance","Reason"]
    cols = [c for c in show if c in df.columns]
    lbls = {"Roster_Shift": "Shift"}
    tc   = "" if em else 'class="t" '
    h    = [f'<table {tc}style="border-collapse:collapse;width:auto;{FONT}"><thead><tr>']
    for c in cols:
        h.append(_thl(lbls.get(c, c)))
    h.append('</tr></thead><tbody>')

    for i, row in df.iterrows():
        atd    = str(row.get("Attendance", ""))
        ba, fa = ACOLORS.get(atd, ("#fff","#000"))
        h.append('<tr>')
        for c in cols:
            v   = row.get(c, "")
            val = (str(v) if v is not None
                   and not (isinstance(v, float) and np.isnan(v))
                   else "&#8212;")
            if c == "Attendance":
                h.append(
                    f'<td style="{TD_S}background:{ba};color:{fa};'
                    f'font-weight:bold;text-align:center;">{val}</td>'
                )
            else:
                h.append(f'<td style="{TD_S}background:#fff;">{val}</td>')
        h.append('</tr>')
    h.append('</tbody></table>')
    return "".join(h)

# ══════════════════════════════════════════════════════════════════════════════
# BUILD ALL SECTIONS
# ══════════════════════════════════════════════════════════════════════════════
def build_all(em=False):
    parts = []
    bi = f"{FONT}font-size:14px;font-weight:bold;color:#fff;margin:0;"
    bs = f"{FONT}font-size:10.5px;color:#fff;margin:3px 0 0;"
    inn = (f'<p style="{bi}">&#128202; Expedia VN Attendance Report — Real-time</p>'
           f'<p style="{bs}">Shift: <strong>{report_shift}</strong> &nbsp;|&nbsp; '
           f'Roster: {roster_date} &nbsp;|&nbsp; '
           f'Generated: {report_now.strftime("%Y-%m-%d %H:%M")} &nbsp;|&nbsp; '
           f'Targets: Planned &le;{TARGET_PLANNED:.0f}% / '
           f'Unplanned &le;{TARGET_UNPLANNED:.0f}% / '
           f'Atd &ge;{TARGET_ATD:.0f}%</p>')
    if em:
        parts.append(
            f'<table width="100%" border="0" cellspacing="0" cellpadding="0" style="margin:0 0 14px;">'
            f'<tr><td style="background:{BANNER_C};padding:10px 14px;border-radius:4px;">'
            f'{inn}</td></tr></table>')
    else:
        parts.append(f'<div style="background:{BANNER_C};padding:10px 14px;'
                     f'border-radius:4px;margin:0 0 14px;">{inn}</div>')

    # ── Spacer helper ─────────────────────────────────────────────────────────
    def spacer():
        if em:
            return ('<table width="100%" border="0" cellspacing="0" cellpadding="0">'
                    '<tr><td style="height:28px;font-size:1px;line-height:1px;">'
                    '&nbsp;</td></tr></table>')
        return '<div style="height:28px;"></div>'

    # ── Section 1 ─────────────────────────────────────────────────────────────
    parts.append(_sec("1","Site wise",
        f"Overall summary — shift {report_shift} on {roster_date}.", em))
    parts.append(render_table(site_sum, SITE_SPEC, em))

    # ── Section 2 ─────────────────────────────────────────────────────────────
    parts.append(spacer())
    parts.append(_sec("2","TL wise",
        "By Team Leader. Login Late = time past shift start (grace 3 min).", em))
    parts.append(render_table(tl_sum, TL_SPEC, em) if not tl_sum.empty
                 else '<p style="color:#888;font-size:11px;">No data</p>')

    # ── Section 3 ─────────────────────────────────────────────────────────────
    parts.append(spacer())
    parts.append(_sec("3","Shift wise",
        f"By roster shift for {roster_date}.", em))
    parts.append(render_table(shift_sum, SH_SPEC, em) if not shift_sum.empty
                 else '<p style="color:#888;font-size:11px;">No data</p>')

    # ── Section 4 ─────────────────────────────────────────────────────────────
    parts.append(spacer())
    parts.append(_sec("4","Absenteeism",
        f"Agents absent/on-leave for {roster_date}. "
        f"Total: <strong>{len(absent_df)}</strong>. "
        f"Sorted: LOB → Supervisor → Shift → Agent.", em))
    parts.append(render_absent(absent_df, em) if not absent_df.empty
                 else f'<p style="color:{MET_FG};font-size:11px;">&#9989; No absenteeism.</p>')

    return "".join(parts)
# ══════════════════════════════════════════════════════════════════════════════
# EMAIL GREETING / SIGNATURE
# ══════════════════════════════════════════════════════════════════════════════
def greeting():
    return f"""
<p style="{FONT}font-size:12px;margin:0 0 10px;line-height:1.7">Dear team,</p>
<p style="{FONT}font-size:12px;margin:0 0 10px;line-height:1.7">
    Please find the Expedia VN Attendance Report till the shift
    <strong>{report_shift}</strong> on <strong>{report_date_s}</strong> (VNT).
</p>
<hr style="border:none;border-top:1px solid #e0e0e0;margin:0 0 12px;">
"""

def signature():
    return f"""
<hr style="border:none;border-top:1px solid #e0e0e0;margin:12px 0 10px;">
<p style="{FONT}font-size:12px;margin:0 0 4px;">Thanks &amp; Regards,</p>
<p style="{FONT}font-size:12px;font-weight:bold;margin:0 0 2px;">Chinh Nguyen</p>
<p style="{FONT}font-size:11px;color:#555;font-weight:bold;margin:0 0 2px;">BI Associate</p>
<p style="{FONT}font-size:11px;color:#555;margin:0 0 2px;line-height:1.6">
    Level 4, Tower 1, OneHub Saigon, Lot C1-2, D1 Street, Saigon Hi Tech Park,<br>
    Tan Phu Ward, District 9, Ho Chi Minh City, Vietnam
</p>
<p style="{FONT}font-size:11px;color:#555;margin:0;">
    Ph No: +84 986 473 419 &nbsp;|&nbsp;
    Email: <a href="mailto:huuchinh.nguyen@concentrix.com"
       style="color:{HDR_MID};font-weight:bold;text-decoration:none;">
       huuchinh.nguyen@concentrix.com</a>
</p>
<p style="{FONT}font-size:10px;color:#aaa;margin-top:8px;">
    Generated: {report_now.strftime("%Y-%m-%d %H:%M")} &nbsp;|&nbsp;
    Source: Master Schedule + hc_extend_combination.parquet + CAPTURE/current_agent
</p>
"""

# ══════════════════════════════════════════════════════════════════════════════
# DISPLAY NOTEBOOK
# ══════════════════════════════════════════════════════════════════════════════
if DISPLAY_NOTEBOOK:
    nb  = ("<!DOCTYPE html><html><head><meta charset='utf-8'>"
           f"<style>{CSS}</style></head><body>"
           + build_all(em=False) + "</body></html>")
    esc = nb.replace("&","&amp;").replace('"',"&quot;").replace("'","&#39;")
    display(HTML(
        f'<iframe srcdoc="{esc}" style="width:100%;border:none;min-height:800px;" '
        f'onload="this.style.height=(this.contentDocument.body.scrollHeight+40)+\'px\'"></iframe>'))
    print("✓ Display done")

# ══════════════════════════════════════════════════════════════════════════════
# SEND EMAIL — reply same-day thread if exists, else send new
# ══════════════════════════════════════════════════════════════════════════════
if SEND_EMAIL:
    import win32com.client
    PREFIX = "Expedia VN Attendance Report as of the shift"

    html_body = (
        "<!--[if mso]><xml><o:OfficeDocumentSettings><o:AllowPNG/>"
        "<o:PixelsPerInch>96</o:PixelsPerInch></o:OfficeDocumentSettings></xml><![endif]-->"
        f"<div style='padding:20px 24px;background:#fff;{FONT}'>"
        + greeting() + build_all(em=True) + signature() + "</div>"
    )

    def send_or_reply(to, cc, subj, body, quit_after=True):
        pythoncom.CoInitialize()
        was_on = any(p.name().lower() == "outlook.exe"
                    for p in psutil.process_iter(["name"]))
        if not was_on:
            for exe in [
                r"C:\Program Files\Microsoft Office\root\Office16\OUTLOOK.EXE",
                r"C:\Program Files (x86)\Microsoft Office\root\Office16\OUTLOOK.EXE",
            ]:
                if os.path.exists(exe): subprocess.Popen([exe]); break
            print("⏳ Starting Outlook...")
            for _ in range(30):
                time.sleep(1)
                try: win32com.client.GetActiveObject("Outlook.Application"); break
                except: pass
        try:
            ol    = win32com.client.Dispatch("Outlook.Application")
            ns    = ol.GetNamespace("MAPI"); ns.Logon()
            sent  = ns.GetDefaultFolder(5)
            items = sent.Items
            items.Sort("[SentOn]", True)

            today_s  = date.today().strftime("%Y-%m-%d")
            yester_s = (date.today() - timedelta(days=1)).strftime("%Y-%m-%d")

            original = None
            for i, item in enumerate(items):
                if i >= 100: break   # giới hạn 100 emails mới nhất
                try:
                    sent_date = item.SentOn.strftime("%Y-%m-%d")
                    if sent_date not in (today_s, yester_s): continue
                    if PREFIX.lower() in item.Subject.lower():
                        original = item
                        break
                except: continue

            if original:
                print(f"✓ Replying to: '{original.Subject}'")
                mail          = original.ReplyAll()
                mail.Subject  = subj
                mail.HTMLBody = body + mail.HTMLBody
                mail.To = to; mail.CC = cc
            else:
                print("✓ No existing thread found — sending new email...")
                mail = ol.CreateItem(0)
                mail.To = to; mail.CC = cc
                mail.Subject = subj; mail.HTMLBody = body

            mail.Send()
            print(f"✓ Sent → {to}")
            time.sleep(3)
        finally:
            if quit_after and not was_on:
                try: ol.Quit(); print("✓ Outlook closed")
                except: pass

    send_or_reply(EMAIL_TO, EMAIL_CC, EMAIL_SUBJECT, html_body)

✓ Report shift   : 2000-0500
✓ Roster date    : 2026-06-03
✓ Shifts in scope: ['0500-1400', '0600-1500', '0700-1600', '0900-1800', '1000-1900', '1100-2000', '1200-2100', '1300-2200', '2000-0500', '2100-0600', '2200-0700', 'AL', 'CO', 'LWP', 'AB', 'SL', 'HAL', 'HLWP', 'HAB', 'HSL', 'CMLF', 'LATE']
✓ Subject        : Expedia VN Attendance Report as of the shift 2000-0500 on 03-Jun-2026 (VNT)
📂 Loading Master Schedule...
✓ Roster agents in scope: 114
📂 Loading HC Extend parquet...
✓ HC Extend rows (latest per agent ≤ 2026-06-03): 881
✓ HC map entries: 881
✓ LOB: {'Lodging': 52, None: 49, 'Non_Lodging': 13}
✓ Leave overrides applied: 0 agent(s)
📂 Loading login CSVs...
✓ Day logins: 0 | Night logins: 10
✓ Agents with login: 8
✓ HC Schedule Day (all shifts): [{'LOB': 'Lodging', 'HC_Schedule_Day': 52.0}, {'LOB': 'Non_Lodging', 'HC_Schedule_Day': 13.0}]
✓ HC Schedule Day Grand Total : 65.0
✓ ATD computed — 114 agents
   PR:8 NCNS:103 Leave:106
✓ Site:3 | TL:9 | Shift:14 | Absent:57


c:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\IPython\core\display.py:431: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


✓ Display done


In [2]:
# # ╔══════════════════════════════════════════════════════════════════════════════╗
# # ║  CELL 2 — TEAMS (HTML → PNG via Chrome headless → Selenium paste)          ║
# # ╚══════════════════════════════════════════════════════════════════════════════╝
# import io, time, tempfile, os, win32clipboard
# import numpy as np
# from PIL import Image
# from selenium import webdriver
# from selenium.webdriver.chrome.options import Options
# from selenium.webdriver.chrome.service import Service
# from selenium.webdriver.common.by import By
# from selenium.webdriver.common.keys import Keys
# from selenium.webdriver.common.action_chains import ActionChains
# from selenium.webdriver.support.ui import WebDriverWait
# from selenium.webdriver.support import expected_conditions as EC
# from selenium.common.exceptions import StaleElementReferenceException
# from webdriver_manager.chrome import ChromeDriverManager

# # ══════════════════════════════════════════════════════════════════════════════
# # CONFIG — TEAMS
# # ══════════════════════════════════════════════════════════════════════════════
# SEND_TEAMS       = True
# TEAMS_GROUP_NAME = "Work together"
# RENDER_WIDTH     = 1100

# # ══════════════════════════════════════════════════════════════════════════════
# # STEP 1 — Render HTML → PNG dùng Chrome headless
# # ══════════════════════════════════════════════════════════════════════════════
# def html_to_pil_image(html_content: str, width: int = 1100) -> Image.Image:
#     full_html = (
#         "<!DOCTYPE html><html><head><meta charset='utf-8'>"
#         f"<style>{CSS}"
#         "body{margin:0;padding:20px;background:#fff;}"
#         "</style></head><body>"
#         + html_content
#         + "</body></html>"
#     )
#     tmp_html = os.path.join(tempfile.gettempdir(), "atd_report_preview.html")
#     with open(tmp_html, "w", encoding="utf-8") as f:
#         f.write(full_html)

#     options = Options()
#     options.add_argument("--headless=new")
#     options.add_argument(f"--window-size={width},900")
#     options.add_argument("--hide-scrollbars")
#     options.add_argument("--disable-gpu")
#     options.add_argument("--no-sandbox")
#     options.add_argument("--disable-dev-shm-usage")

#     driver = webdriver.Chrome(
#         service=Service(ChromeDriverManager().install()),
#         options=options
#     )
#     try:
#         driver.get(f"file:///{tmp_html.replace(os.sep, '/')}")
#         time.sleep(2)
#         height = driver.execute_script("return document.body.scrollHeight")
#         driver.set_window_size(width, height + 40)
#         time.sleep(0.5)
#         png_bytes = driver.get_screenshot_as_png()
#     finally:
#         driver.quit()

#     img = Image.open(io.BytesIO(png_bytes))
#     print(f"✓ Rendered: {img.size[0]}×{img.size[1]}px")
#     return img

# # ══════════════════════════════════════════════════════════════════════════════
# # STEP 2 — Copy PIL Image vào Windows Clipboard
# # ══════════════════════════════════════════════════════════════════════════════
# def copy_image_to_clipboard(image: Image.Image):
#     output = io.BytesIO()
#     image.convert("RGB").save(output, "BMP")
#     data = output.getvalue()[14:]
#     output.close()
#     win32clipboard.OpenClipboard()
#     win32clipboard.EmptyClipboard()
#     win32clipboard.SetClipboardData(win32clipboard.CF_DIB, data)
#     win32clipboard.CloseClipboard()
#     print("✓ Image copied to clipboard")

# # ══════════════════════════════════════════════════════════════════════════════
# # STEP 3 — Selenium paste ảnh vào Teams group chat
# # ══════════════════════════════════════════════════════════════════════════════
# def paste_image_to_teams(group_name: str, image: Image.Image,
#                          caption_lines: list):
#     copy_image_to_clipboard(image)

#     chrome_options = Options()
#     chrome_options.add_argument(r"--user-data-dir=C:/temp/new_chrome_profile")
#     chrome_options.add_argument(r"--profile-directory=Default")
#     chrome_options.add_argument("--start-maximized")

#     driver = webdriver.Chrome(
#         service=Service(ChromeDriverManager().install()),
#         options=chrome_options
#     )
#     wait = WebDriverWait(driver, 25)

#     try:
#         driver.get("https://teams.microsoft.com/")
#         time.sleep(8)

#         # Tìm và click group chat
#         clicked = False
#         for attempt in range(3):
#             try:
#                 chat = wait.until(EC.element_to_be_clickable(
#                     (By.XPATH, f"//span[contains(text(),'{group_name}')]")))
#                 chat.click()
#                 clicked = True
#                 print(f"✓ Entered group: {group_name}")
#                 break
#             except StaleElementReferenceException:
#                 time.sleep(2)
#             except Exception as e:
#                 print(f"❌ Cannot find group '{group_name}': {e}")
#                 return

#         if not clicked:
#             print("❌ Failed after 3 attempts")
#             return

#         time.sleep(3)

#         # Click chat box
#         chat_box = wait.until(EC.element_to_be_clickable(
#             (By.CSS_SELECTOR, "div[role='textbox']")))
#         chat_box.click()

#         # Gõ caption — xuống dòng bằng Shift+Enter
#         actions = ActionChains(driver)
#         for i, line in enumerate(caption_lines):
#             actions.send_keys(line)
#             if i < len(caption_lines) - 1:
#                 actions.key_down(Keys.SHIFT).send_keys(Keys.ENTER).key_up(Keys.SHIFT)
#         actions.perform()
#         time.sleep(1)

#         # Paste ảnh từ clipboard
#         ActionChains(driver).key_down(Keys.CONTROL).send_keys("v").key_up(Keys.CONTROL).perform()
#         print("⏳ Uploading image...")
#         time.sleep(6)

#         # Gửi
#         ActionChains(driver).send_keys(Keys.ENTER).perform()
#         print("✓ Message sent to Teams!")
#         time.sleep(3)

#     except Exception as e:
#         print(f"❌ Error: {e}")
#     finally:
#         driver.quit()
#         print("✓ Chrome closed")

# # ══════════════════════════════════════════════════════════════════════════════
# # MAIN
# # ══════════════════════════════════════════════════════════════════════════════
# if SEND_TEAMS:
#     print("🎨 Rendering Attendance report via Chrome headless...")
#     img = html_to_pil_image(build_all(em=False), width=RENDER_WIDTH)

#     caption_lines = [
#         f"📊 Attendance Realtime Report — updated to shift {report_shift} on {report_date_s}",
#     ]

#     paste_image_to_teams(TEAMS_GROUP_NAME, img, caption_lines)